In [17]:
%load_ext autoreload
%autoreload 2

In [18]:
import pandas as pd
import numpy as np

In [19]:
import sqlalchemy as sa
from sqlalchemy.orm import Session
from crmprtd import setup_logging
from pycds import Station, History

save_path = './comparison_forms/'

db_url = "postgresql+psycopg2://tongli1997@/crmp?host=pg01.pcic.uvic.ca,pg02.pcic.uvic.ca&port=5432,5432&target_session_attrs=read-write&passfile=/workspaces/crmprtd/.pgpass"
log_file_path = save_path

engine = sa.create_engine(db_url, echo=False)
session = Session(engine)

session

In [20]:
path = '/workspaces/crmprtd/update_sheet/'
df = pd.read_excel(path + '20250918-Metadata-AllRequiredChanges.xlsx', header = 1)   # pandas automatically uses openpyxl
df_name_loc = df[
    (df["ISSUE"].str.strip() == "Name, Location") &
    (df["Network ID"] != 11)
]
df_name_loc =  df_name_loc[['Station ID', 'Unique Names','Unique Location (String)', 'Native ID']].reset_index(drop=True)
df_name_loc["Station ID"] = pd.to_numeric(df_name_loc["Station ID"], errors="coerce").astype("Int64")

df_name_loc


,Station ID,Unique Names,Unique Location (String),Native ID
0,13026,-> Similkameen Falls,"120.56784 W, 49.16609 N, Elev. null m ->120.56...",15394
1,13016,-> Mad River,"119.56575 W, 51.66977 N, Elev. null m -> 119.5...",21192
2,13017,-> Eholt,"118.53021 W, 49.14818 N, Elev. null m -> 118.5...",33095
3,13018,-> Bull Mountain Road,"122.11796 W, 52.24473 N, Elev. null m -> 122.1...",47094
4,13019,-> Gitanyow,"128.05074 W, 55.24679 N, Elev. null m -> 128.0...",51193
...,...,...,...,...
73,2477,Williston @ Lost Cabin Ck -> Williston Lake at...,"123.7467167 W, 56.05063333 N, Elev. 712 m -> 1...",LST
74,2508,Wolf R. Upper -> Wolf River Upper,"125.7416667 W, 49.68055556 N, Elev. 1490 m -> ...",WOL
75,13008,-> Enderby 2,"118.9268 W, 50.6529 N, Elev. null m -> 118.926...",1F04P2
76,13009,-> Shovelnose,"120.8629 W, 49.856 N, Elev. null m -> 120.8628...",1C29P


In [21]:
import re


def _parse_single_location(loc_str):
    """
    Parse:
    '123.764438 W, 48.51616 N, Elev. 512.14 m'
    '123.764438 W, 48.51616 N, Elev. null m'

    Returns: (lat, lon, elev)
    """
    if pd.isna(loc_str):
        return np.nan, np.nan, np.nan

    pattern = (
        r"([\d\.]+)\s*([EW]),\s*"
        r"([\d\.]+)\s*([NS]),\s*"
        r"Elev\.\s*(null|[\d\.]+)\s*m"
    )

    m = re.search(pattern, loc_str, re.IGNORECASE)
    if not m:
        return np.nan, np.nan, np.nan

    lon_val, lon_dir, lat_val, lat_dir, elev_val = m.groups()

    lon = float(lon_val) * (-1 if lon_dir.upper() == "W" else 1)
    lat = float(lat_val) * (-1 if lat_dir.upper() == "S" else 1)

    elev = np.nan if elev_val.lower() == "null" else float(elev_val)

    return lat, lon, elev


def parse_old_new_lat_lon_elev(loc_str):
    """
    Returns:
    (old_lat, old_lon, old_elev, new_lat, new_lon, new_elev)
    """
    if pd.isna(loc_str):
        return (np.nan,) * 6

    if "->" in loc_str:
        old_str, new_str = map(str.strip, loc_str.split("->", 1))
    else:
        old_str = new_str = loc_str.strip()

    old_lat, old_lon, old_elev = _parse_single_location(old_str)
    new_lat, new_lon, new_elev = _parse_single_location(new_str)

    return (
        old_lat, old_lon, old_elev,
        new_lat, new_lon, new_elev
    )

cols = [
    'old_lat', 'old_lon', 'old_elev',
    'new_lat', 'new_lon', 'new_elev'
]

df_name_loc[cols] = df_name_loc['Unique Location (String)'].apply(
    lambda x: pd.Series(parse_old_new_lat_lon_elev(x))
)

# 
df_name_loc = df_name_loc.drop(columns=['Unique Location (String)'])

In [22]:


def split_station_name(name):
    if pd.isna(name):
        return pd.Series([None, None, 0])

    parts = [p.strip() for p in name.split("->")]

    if len(parts) == 2:
        old_name, new_name = parts
        n_names = 2
    else:
        old_name = new_name = parts[0]
        n_names = 1

    return pd.Series([old_name, new_name, n_names])
    
df_new = df_name_loc.copy()

df_new[['old_name', 'new_name', 'n_names']] = (
    df_new['Unique Names'].apply(split_station_name)
)

df_new = df_new.drop(columns= 'Unique Names')

In [25]:
df_new.iloc[0:60]

,Station ID,Native ID,old_lat,old_lon,old_elev,new_lat,new_lon,new_elev,old_name,new_name,n_names
0,13026,15394,49.166090,-120.567840,NaN,49.166090,-120.567840,988.0,,Similkameen Falls,2
1,13016,21192,51.669770,-119.565750,NaN,51.669770,-119.565750,545.0,,Mad River,2
2,13017,33095,49.148180,-118.530210,NaN,49.148180,-118.530210,998.0,,Eholt,2
3,13018,47094,52.244730,-122.117960,NaN,52.244730,-122.117960,799.0,,Bull Mountain Road,2
4,13019,51193,55.246790,-128.050740,NaN,55.246790,-128.050740,320.0,,Gitanyow,2
5,13020,53131,57.304770,-130.250560,NaN,57.304770,-130.250560,820.0,,North Burrage,2
6,1658,74,49.098300,-121.663300,295.0,49.095300,-121.662800,320.0,CHILLIWACK NURSERY,CHILLIWACK NURSERY,1
7,1675,101,51.593300,-126.451700,366.0,51.593100,-126.450800,344.0,MACHMELL (MB),MACHMELL,2
8,1683,113,56.983300,-130.253300,609.0,56.980900,-130.251000,605.0,BOB QUINN LK,BOB QUINN LAKE,2
9,1710,145,56.977700,-125.176000,1213.0,57.019400,-125.180400,680.0,INGENIKA PT,INGENIKA POINT,2


In [ ]:
check_sql = sa.text("""
SELECT
    station_id,
    station_name,
    lat,
    lon,
    elev
FROM meta_history
WHERE station_id = :station_id
""")


with engine.begin() as conn:
    for _, row in df_new.iterrows():

        station_id = int(row["Station ID"])

        old_lat  = row["old_lat"]
        old_lon  = row["old_lon"]
        old_elev = row["old_elev"]
        old_name = row["old_name"]

        new_lat  = row["new_lat"]
        new_lon  = row["new_lon"]
        new_elev = row["new_elev"]
        new_name = row["new_name"]

        # --- 1. Read current DB values ---
        db_row = conn.execute(
            check_sql,
            {"station_id": station_id}
        ).fetchone()

        if db_row is None:
            print(f"❌ Station {station_id} not found in DB")
            continue

        # --- 2. Compare with expected OLD values ---
        def norm(x):
            if pd.isna(x):
                return None
            return round(float(x), 6)

        def same(a, b):
            return norm(a) == norm(b)

        if not (
            same(db_row.lat,  old_lat) and
            same(db_row.lon,  old_lon) and
            same(db_row.elev, old_elev)
        ) and db_row.station_name != old_name:
            print(
                f"⚠️ Station {station_id} skipped (DB, XLS values differ)\n"
                f"   DB : name = {db_row.station_name}, lat={db_row.lat}, lon={db_row.lon}, elev={db_row.elev}\n"
                f"   XLS: name = {old_name}, lat={old_lat}, lon={old_lon}, elev={old_elev}"
            )

        else:
            print(
                f"✅ Station {station_id}, all match \n"
                f"   DB : name = {db_row.station_name}, lat={db_row.lat}, lon={db_row.lon}, elev={db_row.elev}"

            )


✅ Station 13026, all match 
   DB : name = None, lat=49.16609, lon=-120.56784, elev=None
✅ Station 13016, all match 
   DB : name = None, lat=51.66977, lon=-119.56575, elev=None
✅ Station 13017, all match 
   DB : name = None, lat=49.14818, lon=-118.53021, elev=None
✅ Station 13018, all match 
   DB : name = None, lat=52.24473, lon=-122.11796, elev=None
✅ Station 13019, all match 
   DB : name = None, lat=55.24679, lon=-128.05074, elev=None
✅ Station 13020, all match 
   DB : name = None, lat=57.30477, lon=-130.25056, elev=None
✅ Station 1658, all match 
   DB : name = CHILLIWACK NURSERY, lat=49.0983, lon=-121.6633, elev=295
✅ Station 1675, all match 
   DB : name = MACHMELL (MB), lat=51.5933, lon=-126.4517, elev=366
✅ Station 1683, all match 
   DB : name = BOB QUINN LK, lat=56.9833, lon=-130.2533, elev=609
✅ Station 1710, all match 
   DB : name = INGENIKA PT, lat=56.9777, lon=-125.176, elev=1213
✅ Station 1722, all match 
   DB : name = VANDERHOOF, lat=54.0555, lon=-124.0102, elev=6

In [32]:
update_sql = sa.text("""
UPDATE meta_history
SET
    station_name  = :new_name,
    lat  = :new_lat,
    lon  = :new_lon,
    elev = :new_elev
WHERE station_id = :station_id
""")


with engine.begin() as conn:
    for _, row in df_new.iterrows():

        station_id = int(row["Station ID"])

        old_lat  = row["old_lat"]
        old_lon  = row["old_lon"]
        old_elev = row["old_elev"]
        old_name = row["old_name"]

        new_lat  = row["new_lat"]
        new_lon  = row["new_lon"]
        new_elev = row["new_elev"]
        new_name = row["new_name"]

        # --- 1. Read current DB values ---
        db_row = conn.execute(
            check_sql,
            {"station_id": station_id}
        ).fetchone()

        if db_row is None:
            print(f"❌ Station {station_id} not found in DB")
            continue

        # --- 3. Perform update ---
        result = conn.execute(
            update_sql,
            {
                "station_id": station_id,
                "new_name": new_name,
                "new_lat": new_lat,
                "new_lon": new_lon,
                "new_elev": new_elev,
            }
        )

        if result.rowcount == 1:
            print(
                f"✅ Updated station {station_id}: "
                f"({old_name}, {old_lat}, {old_lon}, {old_elev}) → "
                f"({new_name}, {new_lat}, {new_lon}, {new_elev})"
            )
        else:
            print(f"⚠️ Station {station_id}: no update applied")


✅ Updated station 13026: (, 49.16609, -120.56784, nan) → (Similkameen Falls, 49.16609, -120.56784, 988.0)
✅ Updated station 13016: (, 51.66977, -119.56575, nan) → (Mad River, 51.66977, -119.56575, 545.0)
✅ Updated station 13017: (, 49.14818, -118.53021, nan) → (Eholt, 49.14818, -118.53021, 998.0)
✅ Updated station 13018: (, 52.24473, -122.11796, nan) → (Bull Mountain Road, 52.24473, -122.11796, 799.0)
✅ Updated station 13019: (, 55.24679, -128.05074, nan) → (Gitanyow, 55.24679, -128.05074, 320.0)
✅ Updated station 13020: (, 57.30477, -130.25056, nan) → (North Burrage, 57.30477, -130.25056, 820.0)
✅ Updated station 1658: (CHILLIWACK NURSERY, 49.0983, -121.6633, 295.0) → (CHILLIWACK NURSERY, 49.0953, -121.6628, 320.0)
✅ Updated station 1675: (MACHMELL (MB), 51.5933, -126.4517, 366.0) → (MACHMELL, 51.5931, -126.4508, 344.0)
✅ Updated station 1683: (BOB QUINN LK, 56.9833, -130.2533, 609.0) → (BOB QUINN LAKE, 56.9809, -130.251, 605.0)
✅ Updated station 1710: (INGENIKA PT, 56.9777, -125.176,

In [33]:
check_sql = sa.text("""
SELECT
    station_id,
    station_name,
    lat,
    lon,
    elev
FROM meta_history
WHERE station_id = :station_id
""")


with engine.begin() as conn:
    for _, row in df_new.iterrows():

        station_id = int(row["Station ID"])

        old_lat  = row["old_lat"]
        old_lon  = row["old_lon"]
        old_elev = row["old_elev"]
        old_name = row["old_name"]

        new_lat  = row["new_lat"]
        new_lon  = row["new_lon"]
        new_elev = row["new_elev"]
        new_name = row["new_name"]

        # --- 1. Read current DB values ---
        db_row = conn.execute(
            check_sql,
            {"station_id": station_id}
        ).fetchone()

        if db_row is None:
            print(f"❌ Station {station_id} not found in DB")
            continue

        # --- 2. Compare with expected OLD values ---
        def norm(x):
            if pd.isna(x):
                return None
            return round(float(x), 6)

        def same(a, b):
            return norm(a) == norm(b)

        if not (
            same(db_row.lat,  new_lat) and
            same(db_row.lon,  new_lon) and
            same(db_row.elev, new_elev)
        ) and db_row.station_name != new_name:
            print(
                f"⚠️ Station {station_id} skipped (DB, XLS values differ)\n"
                f"   DB : name = {db_row.station_name}, lat={db_row.lat}, lon={db_row.lon}, elev={db_row.elev}\n"
                f"   XLS: name = {new_name}, lat={new_lat}, lon={new_lon}, elev={new_elev}"
            )

        else:
            print(
                f"✅ Station {station_id}, all match \n"
                f"   DB : name = {db_row.station_name}, lat={db_row.lat}, lon={db_row.lon}, elev={db_row.elev}"

            )


✅ Station 13026, all match 
   DB : name = Similkameen Falls, lat=49.16609, lon=-120.56784, elev=988.0
✅ Station 13016, all match 
   DB : name = Mad River, lat=51.66977, lon=-119.56575, elev=545.0
✅ Station 13017, all match 
   DB : name = Eholt, lat=49.14818, lon=-118.53021, elev=998.0
✅ Station 13018, all match 
   DB : name = Bull Mountain Road, lat=52.24473, lon=-122.11796, elev=799.0
✅ Station 13019, all match 
   DB : name = Gitanyow, lat=55.24679, lon=-128.05074, elev=320.0
✅ Station 13020, all match 
   DB : name = North Burrage, lat=57.30477, lon=-130.25056, elev=820.0
✅ Station 1658, all match 
   DB : name = CHILLIWACK NURSERY, lat=49.0953, lon=-121.6628, elev=320.0
✅ Station 1675, all match 
   DB : name = MACHMELL, lat=51.5931, lon=-126.4508, elev=344.0
✅ Station 1683, all match 
   DB : name = BOB QUINN LAKE, lat=56.9809, lon=-130.251, elev=605.0
✅ Station 1710, all match 
   DB : name = INGENIKA POINT, lat=57.0194, lon=-125.1804, elev=680.0
✅ Station 1722, all match 
  